# APIM ❤️ Microsoft Foundry

## AI Gateway Sandbox lab
![flow](../../images/ai-gateway.gif)

A hands-on sandbox for how Azure API Management, acting as an **AI Gateway**, governs many agents and users that share **AI models** (Microsoft Foundry), **MCP servers** and **A2A agents**: **identity** (Entra ID + subscription keys), **plans** (entitlements, limits and one budget per consumer), **routing** (regional load balancing and failover), **metering** (cost and tokens per call, chargeback) and **evidence** (traces, logs and metrics in Azure Monitor).

Every **agent** gets its own APIM subscription (its identity and key). Each agent belongs to a **plan** (tier = APIM product). One product covers all three AI Gateway surfaces, and its policy enforces:

| Surface | Control | Policy | Result when exceeded |
|---|---|---|---|
| 🧠 AI models | 🧭 **Model governance**: which models a plan can use | `choose` + `set-body` | premium models are **downgraded** to a cheaper model, or **denied** (403) |
| 🧠 AI models | ✂️ **Output cap**: maximum `max_tokens` per call | `set-body` | the request is capped transparently |
| 🧠 AI models | ⏱️ **Tokens per minute** and 📦 **token quota** | [`llm-token-limit`](https://learn.microsoft.com/azure/api-management/llm-token-limit-policy) | 429 + `Retry-After` / 403 |
| 🛠️ MCP server | 🔐 **Tool entitlement** and ⏱️ **tool calls per minute** | `choose` on the JSON-RPC tool name + [`rate-limit-by-key`](https://learn.microsoft.com/azure/api-management/rate-limit-by-key-policy) | 403 `tool-denied` / 429 |
| 🤝 A2A agent | 🔐 **Agent entitlement** and ⏱️ **tasks per minute** | `choose` + `rate-limit-by-key` | 403 `agent-denied` / 429 |
| All | 💵 **$ budget** across models, tools and agents | [`quota-by-key`](https://learn.microsoft.com/azure/api-management/quota-by-key-policy) incremented by the call's cost | 403 |

Every call is **priced at the gateway** from price lists in named values: `model-pricing` (per 1M tokens), `tool-pricing` (per MCP `tools/call`) and `agent-pricing` (fee per A2A task). The **Sourcing Agent** (an A2A agent on Azure Container Apps) calls the MCP tools and a Foundry model through the same gateway, and the gateway **charges its spend back to the agent that sent the task**. Cost and tokens are emitted to **Application Insights** with the `Agent`, `Tier`, `Model`, `Surface` and `Via` dimensions ([`llm-emit-token-metric`](https://learn.microsoft.com/azure/api-management/llm-emit-token-metric-policy) and [`emit-metric`](https://learn.microsoft.com/azure/api-management/emit-metric-policy)). The lab deploys an **Azure Monitor workbook** and a lightweight **web UI** to drive the demo live.

**Zero trust and resilience.** Every call also needs a **Microsoft Entra ID token** ([`validate-azure-ad-token`](https://learn.microsoft.com/azure/api-management/validate-azure-ad-token-policy)): the key says *who pays*, the token says *who calls*. The Sourcing Agent uses its **managed identity**, and the gateway calls Foundry with its own managed identity. Model calls are **load balanced** across two Foundry regions (priority backend pool + circuit breaker + `retry`), and every policy decision can be seen in an **APIM request trace** and in the **Log Analytics** gateway, LLM and MCP log tables.

| Agent | Plan | Models | Disallowed models | MCP tools | A2A agents | TPM | Max output | $ budget | $ per session |
|---|---|---|---|---|---|---|---|---|---|
| Customer Support Agent (`support-agent`) | 🥇 Gold | gpt-4.1, gpt-4.1-mini, gpt-4.1-nano, DeepSeek-V3.2 | n/a | all 3 | sourcing-agent | 20,000 | 2,000 | $5.00 | $0.02 |
| Research Agent (`research-agent`) | 🥈 Silver | gpt-4.1-mini, gpt-4.1-nano, DeepSeek-V3.2 | downgraded to gpt-4.1-mini | search-products, check-inventory | sourcing-agent | 12,000 | 800 | $0.01 | $0.01 |
| Marketing Copilot (`marketing-agent`) | 🥉 Bronze | gpt-4.1-nano | denied (403) | search-products | none | 1,500 | 300 | $0.02 | $0.01 |

> The Silver and Bronze budgets are deliberately tiny so you can exhaust them live during a demo. Use **Reset budgets** (last step or the UI) to start over.

**FinOps chargeback by user and session.** Four **demo users** (Alice and Ben in Customer Care, Chloe in Marketing, Dev in Supply Chain) each get their own APIM subscription key on a plan, so the demo needs no Entra ID accounts. Every request can carry an `x-session-id` header (or an MCP session / A2A context id). The plan policy caps the $ spend **per session** with a second `quota-by-key`, and the [`chargeback-record`](chargeback-fragment.xml) fragment writes one **chargeback record** (an Application Insights trace) per priced call with the user, team, cost center (from the `chargeback-directory` named value), session, surface and cost. Spend an A2A agent makes for a user carries the same user and session, so a session total is the fully loaded cost of that piece of work. Custom metrics allow at most 5 dimensions, which is why per-user and per-session data is written as traces.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/): run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Adjust the location, models, tiers and agents to your preferences. Check the [model availability by region](https://learn.microsoft.com/azure/ai-foundry/openai/concepts/models).
- ⚠️ The prices are **demo prices** in USD per 1M tokens. Check the [Azure OpenAI pricing](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/) and the Foundry Models pricing for the current prices in your region.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "swedencentral"

aiservices_config = [{"name": "foundry1", "location": "swedencentral", "priority": 1},   # primary region
                     {"name": "foundry2", "location": "francecentral", "priority": 2}]   # failover region (same models)

# Model catalog with demo prices (USD per 1M tokens) used by the gateway to price every call
# primaryCapacity: gpt-4.1-nano gets only 1K TPM in the primary region so a short burst shows a real regional 429 and failover
models_config = [
    {"name": "gpt-4.1",       "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "inputPrice": 2.00, "outputPrice": 8.00},
    {"name": "gpt-4.1-mini",  "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "inputPrice": 0.40, "outputPrice": 1.60},
    {"name": "gpt-4.1-nano",  "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "primaryCapacity": 1, "inputPrice": 0.10, "outputPrice": 0.40},
    {"name": "DeepSeek-V3.2", "publisher": "DeepSeek", "version": "1",          "sku": "GlobalStandard", "capacity": 50, "inputPrice": 0.58, "outputPrice": 1.68},
]

# Plans (APIM products): one plan governs models, MCP tools and A2A agents under one $ budget
tiers_config = [
    {"name": "gold", "displayName": "Gold", "description": "Premium plan: every model, every MCP tool (incl. premium data), the Sourcing Agent, generous limits",
     "allowedModels": ["gpt-4.1", "gpt-4.1-mini", "gpt-4.1-nano", "DeepSeek-V3.2"], "fallbackModel": "gpt-4.1-mini", "onDisallowedModel": "downgrade",
     "maxOutputTokens": 2000, "tpm": 20000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "budgetUsd": 5.00, "sessionBudgetUsd": 0.02,
     "allowedTools": ["search-products", "check-inventory", "get-supplier-quote"], "toolCallsPerMinute": 60,
     "allowedAgents": ["sourcing-agent"], "agentCallsPerMinute": 10},
    {"name": "silver", "displayName": "Silver", "description": "Standard plan: cost-efficient models, standard MCP tools, the Sourcing Agent; premium requests are downgraded",
     "allowedModels": ["gpt-4.1-mini", "gpt-4.1-nano", "DeepSeek-V3.2"], "fallbackModel": "gpt-4.1-mini", "onDisallowedModel": "downgrade",
     "maxOutputTokens": 800, "tpm": 12000, "tokenQuota": 200000, "tokenQuotaPeriod": "Monthly", "budgetUsd": 0.01, "sessionBudgetUsd": 0.01,
     "allowedTools": ["search-products", "check-inventory"], "toolCallsPerMinute": 30,
     "allowedAgents": ["sourcing-agent"], "agentCallsPerMinute": 3},
    {"name": "bronze", "displayName": "Bronze", "description": "Low-cost plan: nano model only, catalog search tool only, no agents, tight token limits",
     "allowedModels": ["gpt-4.1-nano"], "fallbackModel": "gpt-4.1-nano", "onDisallowedModel": "deny",
     "maxOutputTokens": 300, "tpm": 1500, "tokenQuota": 20000, "tokenQuotaPeriod": "Daily", "budgetUsd": 0.02, "sessionBudgetUsd": 0.01,
     "allowedTools": ["search-products"], "toolCallsPerMinute": 10,
     "allowedAgents": [], "agentCallsPerMinute": 1},
    # Internal plan used by platform agents (the Sourcing Agent) to call models and tools. Their spend is charged
    # back to the subscription that invoked the agent, so this budget is only a safety net.
    {"name": "agent-platform", "displayName": "Agent Platform (internal)", "description": "Internal plan for platform agents: their model and tool calls are charged back to the calling subscription",
     "allowedModels": ["gpt-4.1-mini", "gpt-4.1-nano"], "fallbackModel": "gpt-4.1-mini", "onDisallowedModel": "downgrade",
     "maxOutputTokens": 600, "tpm": 50000, "tokenQuota": 2000000, "tokenQuotaPeriod": "Monthly", "budgetUsd": 5.00, "sessionBudgetUsd": 1.00,
     "allowedTools": ["search-products", "check-inventory", "get-supplier-quote"], "toolCallsPerMinute": 120,
     "allowedAgents": [], "agentCallsPerMinute": 1, "internal": True},
]
budget_period_seconds = 30 * 24 * 3600 # the $ budgets renew every 30 days; sessionBudgetUsd caps what one session (x-session-id) can spend

# MCP tool price list (USD per tools/call) and A2A agent fees (USD per task, on top of the agent's own model and tool spend)
tools_config = [
    {"name": "search-products",    "priceUsd": 0.0005, "description": "Search the product catalog"},
    {"name": "check-inventory",    "priceUsd": 0.0002, "description": "Stock level of a SKU per warehouse"},
    {"name": "get-supplier-quote", "priceUsd": 0.0020, "description": "Premium: live supplier quotes for a SKU and quantity"},
]
a2a_agents_config = [
    {"name": "sourcing-agent", "feeUsd": 0.0050, "description": "Sources a product: searches the catalog, checks stock, gets supplier quotes and recommends a supplier"},
]

# Apps / agents and demo users: every entry is an APIM subscription (one key) on its plan. team and costCenter feed the
# chargeback-directory named value, so spend can be rolled up by team, cost center, user and session.
agents_config = [
    {"name": "support-agent",   "displayName": "Customer Support Agent", "tier": "gold",   "kind": "agent", "team": "Customer Care", "costCenter": "CC-1001"},
    {"name": "research-agent",  "displayName": "Research Agent",         "tier": "silver", "kind": "agent", "team": "Supply Chain",  "costCenter": "CC-2002"},
    {"name": "marketing-agent", "displayName": "Marketing Copilot",      "tier": "bronze", "kind": "agent", "team": "Marketing",     "costCenter": "CC-3003"},
    # Demo users: one subscription key per user stands in for an Entra ID identity (no real accounts needed)
    {"name": "user-alice", "displayName": "Alice Chen",   "tier": "gold",   "kind": "user", "team": "Customer Care", "costCenter": "CC-1001"},
    {"name": "user-ben",   "displayName": "Ben Okafor",   "tier": "silver", "kind": "user", "team": "Customer Care", "costCenter": "CC-1001"},
    {"name": "user-chloe", "displayName": "Chloe Martin", "tier": "bronze", "kind": "user", "team": "Marketing",     "costCenter": "CC-3003"},
    {"name": "user-dev",   "displayName": "Dev Patel",    "tier": "gold",   "kind": "user", "team": "Supply Chain",  "costCenter": "CC-2002"},
]

apim_sku = 'Basicv2'
agent_location = "westeurope"  # Azure Container Apps region for the Sourcing Agent (pick a region with Container Apps capacity)
inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "AzureOpenAIV1"  # OpenAI v1 compatible API: /inference/openai/v1/chat/completions
foundry_project_name = deployment_name

# Optional: also host the demo UI on Azure App Service (B1) behind Easy Auth, so only users of this Microsoft Entra tenant can open it.
# Creates an app registration in the tenant (needs permission to register applications). See step 1️⃣4️⃣.
host_demo_ui = False

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

Deployed resources: Log Analytics, Application Insights (custom metrics with dimensions), API Management, two Microsoft Foundry resources (primary and failover region) with the model deployments behind a **priority backend pool with a circuit breaker**, the inference API with the [API policy](policy.xml), the [Entra ID policy fragment](entra-identity-fragment.xml), the **Commerce Tools** REST API and its **MCP server** ([MCP policy](src/tools/mcp-policy.xml)), the **Sourcing Agent** on Azure Container Apps with a user-assigned managed identity, published as an **A2A agent** API ([A2A policy](src/agent/a2a-policy.xml)), one product per plan with the [plan policy](product-policy.xml), one subscription per agent and the **AI Gateway Sandbox** workbook.

In [ ]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Price lists used by the gateway
model_pricing = ";".join(f"{m['name']}={m['inputPrice']:.2f}/{m['outputPrice']:.2f}" for m in models_config)  # USD per 1M input/output tokens
tool_pricing = ";".join(f"{t['name']}={int(round(t['priceUsd'] * 1_000_000))}" for t in tools_config)          # micro-USD per tools/call
agent_pricing = ";".join(f"{a['name']}={int(round(a['feeUsd'] * 1_000_000))}" for a in a2a_agents_config)     # micro-USD per A2A task

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": [{k: v for k, v in m.items() if k not in ("inputPrice", "outputPrice")} for m in models_config] },
        "modelPricing": { "value": model_pricing },
        "toolPricing": { "value": tool_pricing },
        "agentPricing": { "value": agent_pricing },
        "agentLocation": { "value": agent_location },
        "tiersConfig": { "value": [{**{k: v for k, v in t.items() if k not in ("budgetUsd", "sessionBudgetUsd")},
                                    "budgetMicroUsd": int(round(t["budgetUsd"] * 1_000_000)),  # budgets are tracked in micro-USD
                                    "sessionBudgetMicroUsd": int(round(t.get("sessionBudgetUsd", t["budgetUsd"]) * 1_000_000)),
                                    "budgetPeriodSeconds": budget_period_seconds} for t in tiers_config] },
        "agentsConfig": { "value": agents_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "hostDemoUi": { "value": host_demo_ui }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the outputs from the Bicep deployment and write the configuration of the demo UI (`src/demo-config.private.config`, which is git-ignored because it contains the agent keys).

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    outputs = output.json_data['properties']['outputs']
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    inference_base_url = utils.get_deployment_output(output, 'inferenceBaseUrl', 'Inference base URL')
    mcp_url = utils.get_deployment_output(output, 'mcpUrl', 'MCP server URL')
    a2a_url = utils.get_deployment_output(output, 'a2aUrl', 'A2A agent URL')
    agent_card_url = utils.get_deployment_output(output, 'agentCardUrl', 'A2A agent card URL')
    app_insights_id = utils.get_deployment_output(output, 'appInsightsId')
    app_insights_app_id = utils.get_deployment_output(output, 'appInsightsAppId', 'Application Insights App Id')
    workbook_id = utils.get_deployment_output(output, 'workbookId')
    log_analytics_workspace_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics workspace Id')
    foundry_backends = outputs['foundryBackends']['value']
    for backend in foundry_backends:
        utils.print_info(f"Backend pool member: {backend['name']} | {backend['location']} | priority {backend['priority']}")
    demo_ui_url, demo_ui_name = outputs['demoUiUrl']['value'], outputs['demoUiName']['value']
    if demo_ui_url:
        utils.print_info(f"Hosted demo UI: {demo_ui_url} (deploy the app in step 14)")
    agent_keys = outputs['agentKeys']['value']
    for agent in agent_keys:
        utils.print_info(f"Agent: {agent['displayName']} ({agent['name']}) | tier: {agent['tier']} | key: ****{agent['key'][-4:]}")

    demo_config = {
        "inferenceBaseUrl": inference_base_url, "mcpUrl": mcp_url, "a2aUrl": a2a_url, "agentCardUrl": agent_card_url,
        "apimServiceId": apim_service_id, "appInsightsId": app_insights_id,
        "appInsightsAppId": app_insights_app_id, "workbookId": workbook_id, "agents": agent_keys,
        "tenantId": outputs['tenantId']['value'], "logAnalyticsWorkspaceId": log_analytics_workspace_id,
        "logAnalyticsResourceId": outputs['logAnalyticsResourceId']['value'],
        "agentIdentityClientId": outputs['agentIdentityClientId']['value'], "foundryBackends": foundry_backends,
        "tiers": tiers_config, "models": [{k: m[k] for k in ("name", "inputPrice", "outputPrice")} for m in models_config],
        "tools": tools_config, "a2aAgents": a2a_agents_config,
    }
    with open(os.path.join("src", "demo-config.private.config"), "w") as f:
        json.dump(demo_config, f, indent=2)
    utils.print_ok("Demo UI configuration written to src/demo-config.private.config")

<a id='4'></a>
### 4️⃣ Smoke test: every agent calls an allowed model

`call_gateway` sends an OpenAI-compatible chat completion through APIM with **two credentials**: the agent's **subscription key** (which plan pays) and a **Microsoft Entra ID token** (who is calling; here your Azure CLI sign-in, audience `https://cognitiveservices.azure.com`). It prints what the gateway did, using the `x-gw-*` response headers set by the policies: served model, tokens, cost, remaining tokens per minute, the Foundry region that served the call (`x-gw-route`) and the Entra ID caller (`x-gw-caller`).

In [ ]:
import requests, time

agents = {agent['name']: agent for agent in agent_keys}
session_log = []
_entra = {}

def entra_token():
    # Microsoft Entra ID token from the Azure CLI sign-in (an approved client application in the entra-identity fragment)
    if _entra.get("expires", 0) < time.time() + 300:
        token = utils.run("az account get-access-token --resource https://cognitiveservices.azure.com", print_command_to_run=False).json_data
        _entra.update(token=token["accessToken"], expires=float(token.get("expires_on") or time.time() + 1800))
    return _entra["token"]

def gateway_headers(agent, entra=True, extra=None):
    headers = {"api-key": agents[agent]['key']}  # the plan that pays
    if entra:
        headers["Authorization"] = "Bearer " + entra_token()  # who is calling
    return {**headers, **(extra or {})}

def error_text(response):
    try:
        data = response.json()
    except ValueError:
        return response.text
    error = data.get("error") if isinstance(data, dict) else None
    return (error.get("message") if isinstance(error, dict) else error) or (data.get("message") if isinstance(data, dict) else "") or response.text

def call_gateway(agent, model, prompt, max_tokens=200, verbose=True, entra=True, extra_headers=None):
    response = requests.post(f"{inference_base_url}/chat/completions", headers=gateway_headers(agent, entra, extra_headers), timeout=120,
                             json={"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]})
    headers = response.headers
    result = {"agent": agent, "tier": agents[agent]['tier'], "status": response.status_code, "requested": model,
              "served": headers.get("x-gw-model-served", ""), "governance": headers.get("x-gw-governance", ""),
              "promptTokens": int(headers.get("x-gw-prompt-tokens", 0)), "completionTokens": int(headers.get("x-gw-completion-tokens", 0)),
              "costUsd": float(headers.get("x-gw-cost-usd", 0)), "remainingTpm": headers.get("x-gw-remaining-tpm", ""),
              "retryAfter": headers.get("Retry-After", ""), "route": headers.get("x-gw-route", ""), "backend": headers.get("x-gw-backend", ""),
              "caller": headers.get("x-gw-caller", ""), "blockedBy": headers.get("x-gw-blocked-by", ""), "session": headers.get("x-gw-session", ""),
              "requestId": headers.get("x-gw-request-id", ""), "traceId": headers.get("apim-trace-id", "")}
    if response.status_code != 200:
        result["error"] = str(error_text(response))
    session_log.append(result)
    if verbose:
        summary = (f"{agent} [{result['tier']}] {model} -> {result['served'] or '-'} | status {response.status_code} | governance: {result['governance'] or '-'} | "
                   f"tokens {result['promptTokens']}+{result['completionTokens']} | cost ${result['costUsd']:.6f} | remaining TPM {result['remainingTpm'] or '-'} | "
                   f"route: {result['route'] or '-'}")
        if response.status_code == 200:
            utils.print_ok(summary)
        else:
            utils.print_error(f"{summary} | blocked by: {result['blockedBy'] or '-'} | {result.get('error', '')[:200]}")
    return result

call_gateway("support-agent", "gpt-4.1", "In one sentence, what is an AI gateway?")
call_gateway("support-agent", "DeepSeek-V3.2", "In one sentence, what is an AI gateway?")
call_gateway("research-agent", "gpt-4.1-mini", "In one sentence, what is an AI gateway?")
call_gateway("marketing-agent", "gpt-4.1-nano", "Give me a 5 word tagline for an AI gateway.")

<a id='5'></a>
### 5️⃣ Model governance: downgrade and deny

- The **Research Agent** (Silver) asks for the premium `gpt-4.1`. The gateway transparently **downgrades** the request to `gpt-4.1-mini` (`x-gw-governance: model-downgraded`).
- The **Marketing Copilot** (Bronze) asks for `gpt-4.1`. The gateway **denies** the request with a 403 before it reaches Foundry.

In [ ]:
prompt = "Explain in two sentences why model routing reduces AI costs."
downgraded = call_gateway("research-agent", "gpt-4.1", prompt)
denied = call_gateway("marketing-agent", "gpt-4.1", prompt)

assert downgraded["status"] == 200 and downgraded["served"] == "gpt-4.1-mini", "Expected the Silver request to be downgraded"
assert denied["status"] == 403, "Expected the Bronze request to be denied"
utils.print_ok("Model governance enforced: Silver downgraded to gpt-4.1-mini, Bronze denied")

<a id='6'></a>
### 6️⃣ Output token cap

The **Marketing Copilot** asks for 4,000 output tokens. The Bronze tier caps `max_tokens` at 300 (`x-gw-governance: output-capped`), which bounds the worst-case cost of every call.

In [ ]:
capped = call_gateway("marketing-agent", "gpt-4.1-nano", "Write a 1500-word essay about the history of marketing.", max_tokens=4000)
assert capped["status"] == 200 and capped["completionTokens"] <= 300 and "output-capped" in capped["governance"]
utils.print_ok(f"Output capped: {capped['completionTokens']} completion tokens (requested 4000, Bronze cap 300)")

<a id='7'></a>
### 7️⃣ Tokens per minute: burst the Bronze agent

The Bronze tier allows 1,500 tokens per minute. A burst of requests is throttled with **429** and a `Retry-After` header.

In [ ]:
burst = [call_gateway("marketing-agent", "gpt-4.1-nano", "Write a 500-word product description for a smart water bottle.", max_tokens=300) for _ in range(10)]
throttled = [r for r in burst if r["status"] == 429]
assert throttled, "Expected at least one 429"
utils.print_ok(f"{len(throttled)} of {len(burst)} requests throttled with 429 (Retry-After: {throttled[0]['retryAfter']} seconds)")

<a id='8'></a>
### 8️⃣ $ budget: exhaust the Research Agent budget

The Silver tier has a **$0.01** budget. Every successful call increments the agent's `quota-by-key` counter by the call's cost in micro-USD, as computed by the gateway. When the budget is spent, the gateway answers **403** and protects you from runaway spend.

In [ ]:
spent = sum(r["costUsd"] for r in session_log if r["agent"] == "research-agent")
for i in range(30):
    result = call_gateway("research-agent", "gpt-4.1-mini", "Write a detailed report about the evolution of large language models.", max_tokens=800)
    spent += result["costUsd"]
    if result["status"] == 429:
        time.sleep(int(result["retryAfter"] or 10))
    if result["status"] == 403:
        break

assert result["status"] == 403, "Expected the budget to be exhausted"
utils.print_ok(f"Budget exhausted after spending ${spent:.6f} of the $0.01 Silver budget")

<a id='9'></a>
### 9️⃣ MCP server: tools priced and entitled per plan

API Management exposes the **Commerce Tools** REST API as an **MCP server**. The same product (plan) that governs the models also decides which tools an agent may call, how many calls per minute it may make and what each `tools/call` costs (named value `tool-pricing`). The cost is charged to the same $ budget.

- Gold calls the premium `get-supplier-quote` tool: **200**, priced at $0.002.
- Silver calls `get-supplier-quote`: **403 tool-denied**, because the tool is not in its plan.
- Silver calls `check-inventory` and Bronze calls `search-products`: **200**.

In [ ]:
def parse_rpc(text):
    # MCP streamable HTTP answers with plain JSON or a Server-Sent Events stream
    try:
        return json.loads(text)
    except ValueError:
        for line in text.splitlines():
            if line.startswith("data:"):
                try:
                    return json.loads(line[5:].strip())
                except ValueError:
                    pass
    return {}

def call_tool(agent, tool, arguments, verbose=True, entra=True):
    headers = gateway_headers(agent, entra, {"Accept": "application/json, text/event-stream"})
    init = requests.post(mcp_url, headers=headers, timeout=60, json={"jsonrpc": "2.0", "id": 1, "method": "initialize",
                         "params": {"protocolVersion": "2025-03-26", "capabilities": {}, "clientInfo": {"name": "sandbox-lab", "version": "1.0"}}})
    if init.headers.get("mcp-session-id"):
        headers["mcp-session-id"] = init.headers["mcp-session-id"]
    requests.post(mcp_url, headers=headers, timeout=60, json={"jsonrpc": "2.0", "method": "notifications/initialized"})
    response = requests.post(mcp_url, headers=headers, timeout=60,
                             json={"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": tool, "arguments": arguments}})
    data = parse_rpc(response.text)
    result = {"agent": agent, "tier": agents[agent]['tier'], "surface": "tool", "status": response.status_code, "tool": tool,
              "costUsd": float(response.headers.get("x-gw-cost-usd", 0)) if response.status_code == 200 else 0.0,
              "remaining": response.headers.get("x-gw-remaining-tool-calls", ""),
              "error": (data.get("error") or {}).get("message", "") if response.status_code != 200 else ""}
    session_log.append(result)
    if verbose:
        summary = f"{agent} [{result['tier']}] tools/call {tool} | status {response.status_code} | cost ${result['costUsd']:.6f} | remaining calls/min {result['remaining'] or '-'}"
        (utils.print_ok if response.status_code == 200 else utils.print_error)(f"{summary} {result['error'][:150]}")
    return result

premium = call_tool("support-agent", "get-supplier-quote", {"sku": "SKU-1042", "quantity": 500})
denied_tool = call_tool("research-agent", "get-supplier-quote", {"sku": "SKU-1042", "quantity": 500})
standard = call_tool("research-agent", "check-inventory", {"sku": "SKU-1042"})
basic = call_tool("marketing-agent", "search-products", {"category": "electronics"})

assert premium["status"] == 200 and denied_tool["status"] == 403 and standard["status"] in (200, 403) and basic["status"] == 200
utils.print_ok("Tool entitlement and pricing enforced per plan (Silver's check-inventory returns 403 only if its $ budget is already exhausted)")

<a id='10'></a>
### 🔟 A2A agent: what a task really costs, and who pays

The **Sourcing Agent** runs on Azure Container Apps and is published through API Management as an **A2A agent** API. It only accepts tasks routed through the gateway (shared secret), and it calls the MCP tools and a Foundry model **through the same gateway** with its own internal platform subscription and its **managed identity** Entra ID token.

The gateway forwards the caller's identity to the agent (`x-gw-on-behalf-of`), so every downstream tool and model call is **charged back to the agent that sent the task**. The caller is billed the agent fee (`agent-pricing`) plus the downstream spend.

In [ ]:
def send_task(agent, text, verbose=True, session=None):
    body = {"jsonrpc": "2.0", "id": 1, "method": "message/send",
            "params": {"message": {"role": "user", "messageId": str(time.time()), "kind": "message", "parts": [{"kind": "text", "text": text}]}}}
    response = requests.post(a2a_url, headers=gateway_headers(agent, extra={"x-session-id": session} if session else None), json=body, timeout=180)
    data = parse_rpc(response.text)
    task = data.get("result", {}) if isinstance(data, dict) else {}
    h = response.headers
    result = {"agent": agent, "tier": agents[agent]['tier'], "surface": "agent", "status": response.status_code,
              "costUsd": float(h.get("x-gw-cost-usd", 0)) if response.status_code == 200 else 0.0,
              "feeUsd": float(h.get("x-gw-agent-fee-usd", 0)), "downstreamUsd": float(h.get("x-gw-downstream-cost-usd", 0)),
              "billedTo": h.get("x-gw-billed-to", ""), "steps": task.get("metadata", {}).get("steps", []),
              "answer": " ".join(p.get("text", "") for a in task.get("artifacts", []) for p in a.get("parts", [])),
              "error": (data.get("error") or {}).get("message", "") if response.status_code != 200 else ""}
    session_log.append(result)
    if verbose:
        if response.status_code == 200:
            utils.print_ok(f"{agent} [{result['tier']}] A2A task | fee ${result['feeUsd']:.6f} + agent spend ${result['downstreamUsd']:.6f} = ${result['costUsd']:.6f} | billed to {result['billedTo']}")
            for s in result["steps"]:
                utils.print_info(f"   {s['type']:5} {s['name']:20} status {s['status']} | ${s.get('costMicroUsd', 0) / 1e6:.6f} | billed to {s.get('billedTo', '-')}")
            print(result["answer"][:400])
        else:
            utils.print_error(f"{agent} [{result['tier']}] A2A task | status {response.status_code} | {result['error'][:200]}")
    return result

card = requests.get(agent_card_url, headers=gateway_headers("support-agent"), timeout=30).json()
utils.print_info(f"Agent card: {card.get('name')} | skills: {[s['name'] for s in card.get('skills', [])]} | url: {card.get('url')}")

task = send_task("support-agent", "Source 500 units of SKU-1042 for the Q3 launch and recommend a supplier.")
denied_task = send_task("marketing-agent", "Source 100 units of SKU-1001.")

assert task["status"] == 200 and all(s.get("billedTo") == "support-agent" for s in task["steps"]), "Expected the agent's spend to be billed to the caller"
assert denied_task["status"] == 403, "Expected Bronze to be denied (the plan does not include A2A agents)"
utils.print_ok("A2A task priced end to end and charged back to the calling agent; Bronze denied")

<a id='11'></a>
### 1️⃣1️⃣ Cost and tokens in Application Insights

The gateway emits the `Prompt Tokens`, `Completion Tokens`, `Total Tokens`, `CostMicroUSD`, `ToolCalls`, `AgentTasks` and `GovernanceEvents` custom metrics with the `Agent` (the payer), `Tier`, `Model` (the model, tool or agent), `Surface` (model, tool or agent) and `Via` (direct, or the A2A agent that made the call on the payer's behalf) dimensions. The APIM request logs, with the subscription (agent) and product (plan), show the 403 and 429 outcomes.

> ⏳ Telemetry takes 1-3 minutes to arrive in Application Insights.

In [ ]:
import pandas as pd

def query_app_insights(query, timespan="PT4H"):
    token = utils.run("az account get-access-token --resource https://api.applicationinsights.io", print_command_to_run=False).json_data["accessToken"]
    response = requests.post(f"https://api.applicationinsights.io/v1/apps/{app_insights_app_id}/query", headers={"Authorization": f"Bearer {token}"},
                             json={"query": query, "timespan": timespan}, timeout=60)
    response.raise_for_status()
    table = response.json()["tables"][0]
    return pd.DataFrame(table["rows"], columns=[c["name"] for c in table["columns"]])

dims = ('| extend Agent = tostring(customDimensions["Agent"]), Tier = tostring(customDimensions["Tier"]), Model = tostring(customDimensions["Model"]), '
        'Surface = coalesce(tostring(customDimensions["Surface"]), "model"), Via = coalesce(tostring(customDimensions["Via"]), "direct")')
cost_query = f'customMetrics | where name == "CostMicroUSD" {dims} | summarize CostUSD = round(sum(valueSum) / 1e6, 6), Calls = sum(valueCount) by Agent, Tier, Surface, Via, Model | order by CostUSD desc'
tokens_query = (f'customMetrics | where name in ("Prompt Tokens", "Completion Tokens") {dims} '
                '| summarize PromptTokens = sumif(valueSum, name == "Prompt Tokens"), CompletionTokens = sumif(valueSum, name == "Completion Tokens") by Agent, Via, Model')
outcomes_query = ('requests | where url has "/openai/" or url has "/commerce-mcp" or url has "/sourcing-agent" '
                  '| extend Agent = tostring(customDimensions["Subscription Name"]), Tier = tostring(customDimensions["Product Name"]) '
                  '| summarize Calls = count() by Agent, Tier, resultCode | order by Agent asc, resultCode asc')

for attempt in range(12):
    cost_df = query_app_insights(cost_query)
    if not cost_df.empty:
        break
    utils.print_info("Waiting for telemetry to arrive in Application Insights...")
    time.sleep(20)

utils.print_ok("Cost by agent and model (from Application Insights)")
display(cost_df)
utils.print_ok("Tokens by agent and model")
display(query_app_insights(tokens_query))
utils.print_ok("Gateway outcomes by agent")
display(query_app_insights(outcomes_query))

#### 📊 Visualize the cost breakdown

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cost_df.pivot_table(index="Agent", columns="Surface", values="CostUSD", aggfunc="sum", fill_value=0).plot(kind="barh", stacked=True, ax=axes[0], title="Cost by agent and surface (USD)")
cost_df.groupby("Surface")["CostUSD"].sum().plot(kind="pie", ax=axes[1], autopct="%1.1f%%", title="Cost by surface (models, MCP tools, A2A agents)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

workbook_url = f"https://portal.azure.com/#@/resource{workbook_id}/workbook"
utils.print_info(f"Open the AI Gateway Sandbox workbook for the full breakdown: {workbook_url}")

<a id='12'></a>
### 1️⃣2️⃣ Policies in action: Entra ID, load balancing and the evidence

The governance above runs on top of **zero-trust and resilience policies**. This section proves each one with live calls, then shows the evidence the gateway leaves behind in an **APIM request trace** and in **Log Analytics** (the same data the Azure portal shows).

| Policy | What it does | Evidence |
|---|---|---|
| [`validate-azure-ad-token`](https://learn.microsoft.com/azure/api-management/validate-azure-ad-token-policy) ([fragment](entra-identity-fragment.xml)) | Requires a Microsoft Entra ID token for this tenant, audience `https://cognitiveservices.azure.com`, from an approved client application (Azure CLI or the Sourcing Agent's managed identity) | `401`, `x-gw-blocked-by`, `LastErrorSource` in `ApiManagementGatewayLogs`, `x-gw-caller` and the `caller` field of the chargeback records |
| [`authentication-managed-identity`](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy) | The gateway calls Foundry with its own identity, so no Foundry key ever leaves Azure | Request trace |
| [Backend pool](https://learn.microsoft.com/azure/api-management/backends#load-balanced-pool) + [circuit breaker](https://learn.microsoft.com/azure/api-management/backends#circuit-breaker) + [`retry`](https://learn.microsoft.com/azure/api-management/retry-policy) | Sends model calls to the primary region (priority 1) and fails over to the second region on `429` or `5xx` | `x-gw-route`, `BackendId` in the gateway log, the `BackendAttempts` metric |
| [`trace`](https://learn.microsoft.com/azure/api-management/trace-policy) | Writes the policy decisions into the request trace | Request trace |

#### a) Entra ID: no token, no access

The same call with a valid subscription key but **without** an Entra ID token is rejected before any model, tool or agent is reached, so it costs nothing.

In [ ]:
no_token = call_gateway("support-agent", "gpt-4.1-mini", "Hello", max_tokens=20, entra=False)
assert no_token["status"] == 401 and no_token["blockedBy"] == "validate-azure-ad-token", "Expected the Entra ID policy to reject the call"
with_token = call_gateway("support-agent", "gpt-4.1-mini", "Hello", max_tokens=20)
assert with_token["status"] == 200
utils.print_ok(f"No token: 401 blocked by {no_token['blockedBy']} (request id {no_token['requestId']}) | with a token: 200 as {with_token['caller']}")

#### b) Load balancing: a real regional failover

`gpt-4.1-nano` has only 1K tokens per minute in the primary region (`primaryCapacity: 1`), so a short burst gets a real `429` from Sweden Central. The `retry` policy sends the call to France Central, and the circuit breaker takes the primary region out of the pool for a minute, so the next calls go straight to the failover region. `x-gw-route` shows every attempt.

In [ ]:
burst = [call_gateway("support-agent", "gpt-4.1-nano", "Write 200 words about API gateways.", max_tokens=250) for _ in range(8)]
failover = next((r for r in burst if ">" in r["route"]), None)
assert all(r["status"] == 200 for r in burst), "Expected every call to succeed thanks to the failover region"
assert any("foundry2" in r["route"] for r in burst), "Expected some calls to be served by the failover region"
if failover:
    utils.print_ok(f"Failover observed: {failover['route']} (request id {failover['requestId']})")
utils.print_ok(f"Served by region: {pd.Series([r['route'].split('>')[-1].strip() for r in burst]).value_counts().to_dict()}")

#### c) Request trace: every policy decision, step by step

API Management can trace a single request: a short-lived debug credential (`listDebugCredentials`) is sent in the `Apim-Debug-Authorization` header, and `listTrace` returns what each policy did. This is the same trace as the **Trace** button in the portal test console.

In [ ]:
arm = "https://management.azure.com"
def arm_token():
    return utils.run("az account get-access-token --resource https://management.azure.com", print_command_to_run=False).json_data["accessToken"]

arm_headers = {"Authorization": "Bearer " + arm_token()}
debug = requests.post(f"{arm}{apim_service_id}/gateways/managed/listDebugCredentials?api-version=2023-05-01-preview", headers=arm_headers, timeout=60,
                      json={"credentialsExpireAfter": "PT1H", "apiId": f"{apim_service_id}/apis/inference-api", "purposes": ["tracing"]})
debug.raise_for_status()
traced = call_gateway("support-agent", "gpt-4.1-mini", "In one sentence, why use an AI gateway?", max_tokens=60,
                      extra_headers={"Apim-Debug-Authorization": debug.json()["token"]})

for attempt in range(6):
    trace_response = requests.post(f"{arm}{apim_service_id}/gateways/managed/listTrace?api-version=2023-05-01-preview",
                                   headers=arm_headers, json={"traceId": traced["traceId"]}, timeout=60)
    if trace_response.status_code == 200:
        break
    time.sleep(3)
trace_response.raise_for_status()

key_sources = {"include-fragment", "validate-azure-ad-token", "trace", "llm-token-limit", "quota-by-key", "set-backend-service", "backend-pool",
               "authentication-managed-identity", "forward-request", "retry", "emit-metric", "llm-emit-token-metric", "set-header"}
rows = [{"section": section, "source": item.get("source"), "elapsed": item.get("elapsed"),
         "data": json.dumps(item.get("data"), ensure_ascii=False)[:160] if isinstance(item.get("data"), (dict, list)) else str(item.get("data"))[:160]}
        for section, items in (trace_response.json().get("traceEntries") or {}).items() for item in items or []]
trace_df = pd.DataFrame([r for r in rows if r["source"] in key_sources and "Expression was successfully evaluated" not in r["data"]])
utils.print_ok(f"Trace {traced['traceId']}: {len(rows)} records, {len(trace_df)} key policy records")
display(trace_df)

#### d) What the logs track: the same requests in Log Analytics

APIM writes every request to the **`ApiManagementGatewayLogs`** table (plan, subscription, backend, status codes and the policy that failed), every model call to **`ApiManagementGatewayLlmLog`** (deployment and tokens) and every MCP call to **`ApiManagementGatewayMCPLog`**. The `x-gw-request-id` response header is the `CorrelationId` in those tables, so any call can be looked up.

> The gateway log records only the **final** backend attempt, so the failover itself is counted by the `BackendAttempts` custom metric (the `AppMetrics` table of the same workspace).

> ⏳ Logs take 2-5 minutes to arrive in Log Analytics.

In [ ]:
def query_log_analytics(query, timespan="PT4H"):
    token = utils.run("az account get-access-token --resource https://api.loganalytics.io", print_command_to_run=False).json_data["accessToken"]
    response = requests.post(f"https://api.loganalytics.io/v1/workspaces/{log_analytics_workspace_id}/query", headers={"Authorization": "Bearer " + token},
                             json={"query": query, "timespan": timespan}, timeout=90)
    response.raise_for_status()
    table = response.json()["tables"][0]
    return pd.DataFrame(table["rows"], columns=[c["name"] for c in table["columns"]])

ids = [r["requestId"] for r in [no_token, failover or burst[-1], traced] if r["requestId"]]
gateway_query = ('ApiManagementGatewayLogs | where CorrelationId in (' + ", ".join(f'"{i}"' for i in ids) + ') '
                 '| project TimeGenerated, CorrelationId, ApiId, ProductId, ApimSubscriptionId, ResponseCode, BackendId, BackendResponseCode, '
                 'LastErrorSource, LastErrorReason, TotalTime')
for attempt in range(18):
    gateway_df = query_log_analytics(gateway_query)
    if len(gateway_df) >= len(ids):
        break
    utils.print_info("Waiting for the gateway logs to arrive in Log Analytics...")
    time.sleep(20)

utils.print_ok("ApiManagementGatewayLogs: the 401, the failover and the traced call")
display(gateway_df)
utils.print_ok("ApiManagementGatewayLlmLog: tokens counted by the gateway per deployment")
display(query_log_analytics('ApiManagementGatewayLlmLog | where isnotempty(DeploymentName) '
                            '| summarize Calls = dcount(CorrelationId), PromptTokens = sum(PromptTokens), CompletionTokens = sum(CompletionTokens) by DeploymentName'))
utils.print_ok("BackendAttempts metric: calls that needed a second region")
display(query_log_analytics('AppMetrics | where Name == "BackendAttempts" | extend Backend = tostring(Properties.Backend), Failover = tostring(Properties.Failover) '
                            '| summarize Requests = sum(ItemCount), Attempts = sum(Sum) by Backend, Failover'))
utils.print_ok("Entra ID rejections by plan")
display(query_log_analytics('ApiManagementGatewayLogs | where LastErrorSource == "validate-azure-ad-token" | summarize Rejected = count() by ProductId, LastErrorReason'))

portal_logs = f"https://portal.azure.com/#@/resource{outputs['logAnalyticsResourceId']['value']}/logs"
utils.print_info(f"Run the same queries in the Azure portal: {portal_logs}")
utils.print_info("The workbook's 'Policies in action' section charts the same data")

<a id='13'></a>
### 1️⃣3️⃣ FinOps chargeback by user and session

The plans answer *how much can this subscription spend*. FinOps also needs to know **who** spent it and **on what piece of work**, so the cost can be charged back to a team and cost center.

| Demo user | Subscription | Plan | Team | Cost center |
|---|---|---|---|---|
| Alice Chen | `user-alice` | 🥇 Gold | Customer Care | CC-1001 |
| Ben Okafor | `user-ben` | 🥈 Silver | Customer Care | CC-1001 |
| Chloe Martin | `user-chloe` | 🥉 Bronze | Marketing | CC-3003 |
| Dev Patel | `user-dev` | 🥇 Gold | Supply Chain | CC-2002 |

- **Users are subscription keys.** Each demo user has their own APIM subscription on a plan, so no Entra ID accounts are needed for the demo. The `chargeback-directory` named value maps each subscription to its team and cost center.
- **Sessions come from the request.** The plan policy resolves the session from the `x-session-id` header, then the MCP `Mcp-Session-Id`, then the A2A `contextId` (values are validated, otherwise `no-session`), and returns it in `x-gw-session`. When an A2A agent works for a user, the gateway forwards the session (`x-gw-on-behalf-of-session`), so the agent's model and tool calls land in the **same user and session**.
- **A $ cap per session.** A second [`quota-by-key`](https://learn.microsoft.com/azure/api-management/quota-by-key-policy), keyed on subscription + session, stops one runaway conversation or agent loop before it eats the whole plan budget (`403`, `x-gw-blocked-by: quota-by-key (session-budget)`). A new session starts fresh. The cap is a soft limit: quota counters are updated asynchronously after each call, so a fast loop can run a few calls past it.
- **Chargeback records.** The [`chargeback-record`](chargeback-fragment.xml) fragment writes one [`trace`](https://learn.microsoft.com/azure/api-management/trace-policy) per priced call to Application Insights (`traces` table) with the user, team, cost center, session, surface, item, tokens and cost. Custom metrics allow at most 5 dimensions, so per-user and per-session data is written as traces, not metric dimensions.

> In production the user would come from the Entra ID token (`oid` or `preferred_username` claim, which `validate-azure-ad-token` already exposes) and the team and cost center from a directory or HR attribute, instead of one subscription key per user.

In [ ]:
import uuid

run_id = uuid.uuid4().hex[:6]
def new_session(name):
    return f"{name}-{run_id}"

# Ordinary work: each user's calls are tagged with a session id
work = [
    call_gateway("user-alice", "gpt-4.1-mini", "Summarize our refund policy in two sentences.", max_tokens=120, extra_headers={"x-session-id": new_session("alice-ticket-4711")}),
    call_gateway("user-alice", "gpt-4.1-mini", "Draft a short apology email for a late delivery.", max_tokens=150, extra_headers={"x-session-id": new_session("alice-ticket-4711")}),
    call_gateway("user-ben", "gpt-4.1-mini", "List three ways to reduce call handling time.", max_tokens=120, extra_headers={"x-session-id": new_session("ben-ticket-4712")}),
    call_gateway("user-chloe", "gpt-4.1-nano", "Give me a tagline for a spring sale.", max_tokens=40, extra_headers={"x-session-id": new_session("chloe-campaign-q3")}),
    call_gateway("user-dev", "DeepSeek-V3.2", "Name two risks of single-sourcing a component.", max_tokens=120, extra_headers={"x-session-id": new_session("dev-sourcing-review")}),
]
assert all(r["status"] == 200 and r["session"] for r in work), "Expected every call to succeed and echo its session"

# An A2A task for Dev: the agent's downstream model and tool calls are charged to Dev's session
dev_task = send_task("user-dev", "Source 200 units of SKU-1042 and recommend a supplier.", session=new_session("dev-sourcing-review"))
assert dev_task["status"] == 200 and all(s.get("billedTo") == "user-dev" for s in dev_task["steps"]), "Expected the agent's spend to be billed to Dev"

# A runaway session: Alice keeps calling the premium model in one session until the $ cap per session stops it
runaway_session = new_session("alice-runaway")
runaway = []
for i in range(30):  # the cap is a soft limit: counters sync asynchronously, so a fast loop can run a few calls past it
    r = call_gateway("user-alice", "gpt-4.1", "Write a detailed 150 word product description for a smart thermostat.", max_tokens=250,
                     extra_headers={"x-session-id": runaway_session}, verbose=False)
    runaway.append(r)
    if r["status"] != 200:
        break
spent = sum(r["costUsd"] for r in runaway)
assert runaway[-1]["status"] == 403 and runaway[-1]["blockedBy"] == "quota-by-key (session-budget)", "Expected the session budget to stop the runaway session"
utils.print_ok(f"Runaway session {runaway_session}: {len(runaway) - 1} calls (${spent:.5f}) then {runaway[-1]['status']} blocked by {runaway[-1]['blockedBy']}")
fresh = call_gateway("user-alice", "gpt-4.1-mini", "Hello", max_tokens=20, extra_headers={"x-session-id": new_session("alice-new-ticket")})
assert fresh["status"] == 200, "Expected a new session to start with a fresh budget"

#### Slice the spend: team, cost center, user and session

Sum the `costMicroUsd` field of the chargeback records. An A2A record carries only the agent fee; the agent's model and tool calls are separate records (`via` = the agent) for the same user and session, so nothing is counted twice.

> ⏳ Traces take 1-3 minutes to arrive in Application Insights.

In [ ]:
chargeback = ('traces | where tostring(customDimensions["record"]) == "chargeback" '
              '| extend Team = tostring(customDimensions["team"]), CostCenter = tostring(customDimensions["costCenter"]), '
              'User = tostring(customDimensions["user"]), DisplayName = tostring(customDimensions["displayName"]), '
              'Kind = tostring(customDimensions["kind"]), Session = tostring(customDimensions["session"]), '
              'Surface = tostring(customDimensions["surface"]), Item = tostring(customDimensions["item"]), Via = tostring(customDimensions["via"]), '
              'Cost = tolong(customDimensions["costMicroUsd"]), Tokens = tolong(customDimensions["promptTokens"]) + tolong(customDimensions["completionTokens"]) ')

sessions_query = (chargeback + f'| where Session endswith "{run_id}" '
                  '| summarize CostUSD = round(sum(Cost) / 1e6, 6), Calls = count(), Tokens = sum(Tokens), Surfaces = make_set(Surface) '
                  'by Team, CostCenter, DisplayName, Session | order by CostUSD desc')
for attempt in range(15):
    sessions_df = query_app_insights(sessions_query)
    if not sessions_df.empty and sessions_df["Session"].nunique() >= 6:
        break
    utils.print_info("Waiting for the chargeback records to arrive in Application Insights...")
    time.sleep(20)

utils.print_ok("Chargeback by team and cost center (users and agents, last 4 hours)")
display(query_app_insights(chargeback + '| summarize CostUSD = round(sum(Cost) / 1e6, 6), Calls = count(), Users = dcountif(User, Kind == "user"), '
                           'Sessions = dcount(Session) by Team, CostCenter | order by CostUSD desc'))
utils.print_ok("Chargeback by user and session (this run)")
display(sessions_df)
utils.print_ok(f"Drill-down: Dev's sourcing session, including the work the Sourcing Agent did on Dev's behalf")
display(query_app_insights(chargeback + f'| where Session == "{new_session("dev-sourcing-review")}" '
                           '| project timestamp, Surface, Item, Via, Tokens, CostUSD = round(Cost / 1e6, 6) | order by timestamp asc'))

sessions_df.pivot_table(index="DisplayName", columns="Session", values="CostUSD", aggfunc="sum", fill_value=0).plot(
    kind="barh", stacked=True, figsize=(12, 4), title="Cost by user and session (USD)")
plt.tight_layout()
plt.show()
utils.print_info("The workbook's 'Chargeback by team, user and session' section and the UI Chargeback tab show the same records")

<a id='14'></a>
### 1️⃣4️⃣ Launch the demo UI

The lightweight web UI ([src/app.py](src/app.py), Python standard library only) lets you pick an agent and send requests to a **model**, an **MCP tool** or the **A2A agent**, run bursts, mixed traffic or the guided scenarios, see what the gateway did for every call, compare the plans, track each agent's $ budget, and query the Application Insights breakdown live. The **Policies & evidence** tab shows the deployed policies, request traces and the Log Analytics evidence, with links to the Azure portal. The **Chargeback** tab lets you act as a demo user in a session and shows the ledger by team, user and session. The UI calls the gateway with an Entra ID token from your Azure CLI sign-in.

Run it from a terminal in this folder to keep it running independently of the notebook:

```bash
python src/app.py --port 8080
```

Or run the next cell to start it in the background, then open http://localhost:8080.

#### Host it for the whole team (optional)

With `host_demo_ui = True`, the deployment also creates an **App Service** (Linux, B1) that runs the same `app.py`, protected by **App Service authentication (Easy Auth)**: only users who sign in to this Microsoft Entra tenant can open it. The sign-in uses a new app registration that authenticates with a **federated identity credential** on the app's managed identity, so there is no client secret to store or rotate. The hosted app calls the gateway, the management API, Log Analytics and Application Insights with that **managed identity**, and the `entra-identity` policy fragment records the signed-in presenter as the caller (`x-gw-caller: you@contoso.com (via demo UI)`).

The cell after next packages `src/app.py`, `src/static` and the generated configuration (which contains the subscription keys and is never served to the browser) and deploys it. Run it again after a redeployment so the hosted app picks up the new keys.

In [ ]:
import subprocess
ui_process = subprocess.Popen([sys.executable, os.path.join("src", "app.py"), "--port", "8080"])
time.sleep(2)
utils.print_ok("Demo UI running on http://localhost:8080 (run ui_process.terminate() to stop it)")

#### Deploy the app to the hosted demo UI (optional)


In [ ]:
import tempfile, zipfile, urllib.request

if not demo_ui_url:
    utils.print_info("The demo UI is not hosted: set host_demo_ui = True in step 0 and run the deployment again.")
else:
    # Package app.py, static/ and the generated configuration (app.py at the root of the zip)
    package = os.path.join(tempfile.gettempdir(), f"{demo_ui_name}.zip")
    with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join("src", "app.py"), "app.py")
        z.write(os.path.join("src", "demo-config.private.config"), "demo-config.private.config")
        for file in os.listdir(os.path.join("src", "static")):
            z.write(os.path.join("src", "static", file), f"static/{file}")
    # --track-status false: the site already tried to start before it had any content, so the CLI status tracking reports a false failure
    utils.run(f"az webapp deploy -g {resource_group_name} -n {demo_ui_name} --src-path {package} --type zip --track-status false -o none",
              f"Deployed the demo UI to {demo_ui_name}", "Failed to deploy the demo UI")
    for attempt in range(40):  # /healthz is the only path that answers without a sign-in
        try:
            if urllib.request.urlopen(f"{demo_ui_url}/healthz", timeout=10).status == 200:
                utils.print_ok(f"Hosted demo UI is running: {demo_ui_url} (sign in with an account of this tenant)")
                break
        except Exception:
            pass
        time.sleep(15)
    else:
        utils.print_error(f"The demo UI did not start: check the logs with az webapp log tail -g {resource_group_name} -n {demo_ui_name}")


<a id='15'></a>
### ↺ Reset the budgets and token quotas

Every counter key includes the `budget-epoch` named value, so updating it starts fresh budgets and token quotas for every agent. Run this before each demo. The UI **Reset budgets** button does the same.

In [ ]:
epoch = str(int(time.time()))
output = utils.run(f"az apim nv update --service-name {apim_service_id.split('/')[-1]} -g {resource_group_name} --named-value-id budget-epoch --value {epoch}",
                   "Budgets and token quotas reset", "Failed to reset the budgets")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.